In [ ]:

# 1. Импорт библиотек

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold



# 2. Загрузка данных

sheet_id = "1q-nbWuFrfrIBMXmZfNW78N3bx5v60Vb9"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"

df = pd.read_csv(url)



# 3. Очистка данных

df = df.drop(columns=['Unnamed: 0'])
df = df.fillna(df.median(numeric_only=True))

selector = VarianceThreshold(threshold=0.01)
X_temp = df.drop(columns=['IC50, mM', 'CC50, mM', 'SI'])

selector.fit(X_temp)

low_var_cols = X_temp.columns[~selector.get_support()]
df = df.drop(columns=low_var_cols)



# 4. Подготовка таргетов


# SI > медианы
si_median = df['SI'].median()
df['SI_class_median'] = (df['SI'] > si_median).astype(int)

# SI > 8
df['SI_class_8'] = (df['SI'] > 8).astype(int)



# 5. Признаки (без утечки)

X = df.drop(columns=[
    'IC50, mM', 'CC50, mM', 'SI',
    'SI_class_median', 'SI_class_8'
])


# 6. SI > медианы

print("=== SI > медианы ===")

X_train, X_test, y_train, y_test = train_test_split(
    X, df['SI_class_median'], test_size=0.2, random_state=42
)

# Logistic
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)

y_pred_lr = logreg.predict(X_test)

print("\nLogistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

# RF
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("\nRandom Forest")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))



# 7. SI > 8

print("\n=== SI > 8 ===")

X_train, X_test, y_train, y_test = train_test_split(
    X, df['SI_class_8'], test_size=0.2, random_state=42
)

# Logistic
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)

y_pred_lr = logreg.predict(X_test)

print("\nLogistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

# RF
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("\nRandom Forest")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

### Что делаю

- Строю две задачи классификации:
   **SI > медианы**
   **SI > 8**

- В качестве признаков использую только химические дескрипторы  
- `IC50, mM` и `CC50, mM` в признаки **не включаю**, потому что `SI` считается через них и иначе будет утечка  

### Что буду смотреть

Для каждой задачи сравниваю две модели:

 **Logistic Regression** — как простой baseline  
 **Random Forest** — как более сильную нелинейную модель  

### Что ожидаю

По предыдущим шагам уже видно, что:

- задачи по **IC50** и **CC50** решаются заметно лучше
- **SI** предсказывается хуже, потому что это производный показатель
- скорее всего:
   Logistic Regression снова даст слабый результат
   Random Forest будет лучше, но качество всё равно окажется ниже, чем в задачах по IC50 и CC50

### Что нужно будет зафиксировать после запуска

Для **SI > медианы**:
- accuracy
- precision / recall / f1-score по двум классам

Для **SI > 8**:
- accuracy
- отдельно качество по классу **1**, потому что здесь уже есть дисбаланс классов

### Что получилось

#### SI > медианы

**Logistic Regression:**
- Accuracy: **0.54**
- Модель предсказывает почти только класс **0**
- Класс 1 не определяется

**Random Forest:**
- Accuracy: **0.68**
- Класс 0:
   precision: 0.69  
  recall: 0.71  
- Класс 1:
   precision: 0.66  
   recall: 0.63  



#### SI > 8

**Logistic Regression:**
- Accuracy: **0.65**
- Модель снова предсказывает почти только класс **0**
- Класс 1 не определяется

**Random Forest:**
- Accuracy: **0.71**
- Класс 0:
   precision: 0.73  
   recall: 0.87  
- Класс 1:
   precision: 0.64  
   recall: 0.42  


### Вывод

- Logistic Regression стабильно не справляется  
- Random Forest даёт:
   ~0.68 для SI > медианы  
   ~0.71 для SI > 8  

- Качество ниже, чем в задачах IC50 и CC50  
- Особенно плохо определяется класс **SI > 8** (recall = 0.42)

 SI даже в классификации предсказывается заметно хуже  

Следующее:финальное сравнение всех моделей, подведение итогов и анализ